# Reproducible MAKD-HAR experiment: WISDM v1.1

This notebook uses subject-disjoint partitions, train-only scaling, five distinct teachers, coalition-based Shapley values, scale-independent teacher weighting, saved per-seed results, and corrected statistical summaries.

Run the quick check first. Never copy quick-check values into the manuscript.


## 1. Prepare Google Drive and the Colab runtime

Before running this section, use this Google Drive layout:

```text
MyDrive/mk-har/
├── revised_pipeline/
├── resources/
│   └── WISDM_ar_v1.1_raw.txt
└── results/
```

The code and raw dataset are copied into Colab's temporary storage for faster access. Experiment results are written directly to Google Drive so completed seeds survive a Colab disconnection.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Persistent locations in Google Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/mk-har')
DRIVE_PROJECT_ROOT = next((p for p in (DRIVE_ROOT, DRIVE_ROOT / 'revised_pipeline') if (p / 'run_experiment.py').is_file()), DRIVE_ROOT)
DRIVE_DATA_FILE = DRIVE_ROOT / 'resources' / 'WISDM_ar_v1.1_raw.txt'
DRIVE_RESULTS_ROOT = DRIVE_ROOT / 'results' / 'wisdm'

# Fast temporary locations in the Colab runtime.
PROJECT_ROOT = Path('/content/revised_pipeline')
DATA_ROOT = Path('/content/WISDM_ar_v1.1_raw.txt')

assert (DRIVE_PROJECT_ROOT / 'run_experiment.py').is_file(), (
    f'Project not found: {DRIVE_PROJECT_ROOT}'
)
assert DRIVE_DATA_FILE.is_file(), f'WISDM raw file not found: {DRIVE_DATA_FILE}'

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(
    DRIVE_PROJECT_ROOT,
    PROJECT_ROOT,
    ignore=shutil.ignore_patterns('results', '__pycache__', '*.pyc'),
)
shutil.copy2(DRIVE_DATA_FILE, DATA_ROOT)

RESULTS_ROOT = DRIVE_RESULTS_ROOT
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

%cd {PROJECT_ROOT}
print(f'Project: {PROJECT_ROOT}')
print(f'Dataset: {DATA_ROOT}')
print(f'Persistent results: {RESULTS_ROOT}')


## 2. Install the declared environment


In [ ]:
%pip install -q -r requirements-colab.txt

# If Colab asks for a runtime restart after installation, restart it and rerun
# Sections 1 and 2 before continuing.


## 3. Verify the runtime and raw dataset


In [ ]:
import sys
import tensorflow as tf

print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('WISDM file size (MB):', round(DATA_ROOT.stat().st_size / (1024 ** 2), 2))
assert (PROJECT_ROOT / 'run_experiment.py').is_file()
assert DATA_ROOT.is_file() and DATA_ROOT.stat().st_size > 0


## 4. Quick structural check

This one-seed run checks WISDM parsing, real 20 Hz window features, subject-disjoint splitting, the five teachers, coalition evaluation, Shapley weighting, and student training. Its values are not manuscript results.


In [ ]:
import subprocess

quick_command = [
    sys.executable, 'run_experiment.py',
    '--dataset', 'wisdm',
    '--data-root', str(DATA_ROOT),
    '--output', str(RESULTS_ROOT / 'quick-check'),
    '--quick',
    '--verbose', '1',
]
subprocess.run(quick_command, check=True)


## 5. Inspect quick-check artifacts

Confirm that the output contains the dataset manifest and seed result. Check that train, validation, and test user lists are disjoint and that the Shapley diagnostics do not silently conceal uniform weighting.


In [ ]:
quick_files = sorted(
    str(path.relative_to(RESULTS_ROOT))
    for path in (RESULTS_ROOT / 'quick-check').rglob('*')
    if path.is_file()
)
print(*quick_files, sep='\n')
assert quick_files, 'The quick check produced no result files.'


## 6. Production experiment

This is the manuscript-producing run: ten seeds, all 32 coalitions for five teachers, early stopping, and resumable outputs. Coalition training is expensive. Completed seed files in Google Drive are skipped automatically when the command is restarted.


In [ ]:
production_command = [
    sys.executable, 'run_experiment.py',
    '--dataset', 'wisdm',
    '--data-root', str(DATA_ROOT),
    '--output', str(RESULTS_ROOT / 'production'),
    '--shapley-mode', 'exact',
    '--teacher-epochs', '60',
    '--student-epochs', '60',
    '--coalition-epochs', '20',
    '--batch-size', '128',
    '--patience', '10',
    '--verbose', '1',
]
subprocess.run(production_command, check=True)


## 7. Generate mean ± SD and corrected significance tests


In [ ]:
subprocess.run([
    sys.executable,
    'summarize_results.py',
    str(RESULTS_ROOT / 'production'),
], check=True)


## 8. Create a shareable production-results archive


In [ ]:
archive_base = Path('/content/wisdm-production-results')
local_archive = Path(shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=RESULTS_ROOT / 'production',
))
drive_archive = RESULTS_ROOT / local_archive.name
shutil.copy2(local_archive, drive_archive)
local_archive.unlink()
print(f'Saved: {drive_archive}')


## Required audit before manuscript use

Confirm ten complete seed files; no NaN values; disjoint user lists in `dataset-manifest.json`; all 32 coalition evaluations per exact-Shapley run; meaningful weighting diagnostics when claiming an adaptive-weighting benefit; and agreement between `summary.csv` and every manuscript table. Energy and MAC values remain absent until measured with validated target-hardware procedures.
